In [5]:
!pip install langchain langchain-community chromadb pypdf sentence-transformers torch transformers accelerate rank_bm25

import os
from typing import List, Dict, Optional
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
import re
import uuid
from rank_bm25 import BM25Okapi


In [3]:
!pip install -U sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 488.0/488.0 kB 9.6 MB/s eta 0:00:00ta 0:00:01
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 4.1.0
    Uninstalling sentence-transformers-4.1.0:
      Successfully uninstalled sentence-transformers-4.1.0


In [13]:
class LegalSearchAgent:
    # =======================================================================
    #                         INITIALIZATION
    # =======================================================================

    def __init__(self, pdf_folder: str, db_path: str = "chroma_db", test_mode: bool = True):
        self.pdf_folder = pdf_folder
        self.db_path = db_path
        self.test_mode = test_mode

        self.vectorstore = None
        self.bm25 = None
        self.bm25_corpus = []
        self.bm25_docs = []

        # ---------------------------------------------------
        # Load Local LLM (Phi-2 or fallback)
        # ---------------------------------------------------
        print("🤖 Loading local LLM (Phi-2)...")
        try:
            model_name = "microsoft/phi-2"
            tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=torch.float32,
                device_map="auto",
                trust_remote_code=True
            )
            pipe = pipeline(
                "text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=512,
                temperature=0.1,
                do_sample=True,
                top_p=0.95
            )
            self.llm = HuggingFacePipeline(pipeline=pipe)
            print("Local LLM loaded successfully.")
        except Exception as e:
            print(f"⚠️ Could not load Phi-2: {e}")
            print("Falling back to extractive answers only.")
            self.llm = None

        # ---------------------------------------------------
        # Load LegalBERT embeddings
        # ---------------------------------------------------
        print("📚 Loading Sentence-BERT embeddings...")
        self.embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/paraphrase-distilroberta-base-v2",
            model_kwargs={'device': 'cuda'}
        )
        print("Sentence-BERT loaded.\n")


    # =======================================================================
    #                         BUILD VECTOR DATABASE
    # =======================================================================

    def build_vectordb(self):
        if os.path.exists(self.db_path):
            print("📚 Loading existing Chroma DB...")
            self.vectorstore = Chroma(
                persist_directory=self.db_path,
                embedding_function=self.embeddings
            )
            print(f"Loaded {self.vectorstore._collection.count()} chunks.")
            
            # Rebuild BM25 index
            print("Rebuilding BM25 index...")
            pdf_files = [f for f in os.listdir(self.pdf_folder) if f.lower().endswith(".pdf") and len(f) > 5]
            all_docs = []
            
            for pdf in pdf_files:
                try:
                    loader = PyPDFLoader(os.path.join(self.pdf_folder, pdf))
                    pages = loader.load()
                    all_docs.extend(pages)
                except Exception as e:
                    print(f"❌ Error loading {pdf}: {e}")
            
            splitter = RecursiveCharacterTextSplitter(
                chunk_size=1000,
                chunk_overlap=200,
                separators=["\n\n", "\n", ".", " ", ""]
            )
            chunks = splitter.split_documents(all_docs)
            
            for ch in chunks:
                self.bm25_docs.append(ch)
                self.bm25_corpus.append(ch.page_content.split())
            
            self.bm25 = BM25Okapi(self.bm25_corpus)
            print("📌 BM25 index rebuilt.\n")
            return

        print("🔨 Building new vector DB...")

        pdf_files = [f for f in os.listdir(self.pdf_folder) if f.lower().endswith(".pdf") and len(f) > 5]

        if self.test_mode:
            pdf_files = pdf_files[:5]
            print(f"⚠️ TEST MODE: Using only first {len(pdf_files)} PDFs")
        else:
            print(f"Found {len(pdf_files)} PDFs to process.")

        all_docs = []

        # ---------------------------------------------------
        # LOAD PDFS
        # ---------------------------------------------------
        for idx, pdf in enumerate(pdf_files, 1):
            try:
                print(f"📄 Loading [{idx}/{len(pdf_files)}]: {pdf}")
                loader = PyPDFLoader(os.path.join(self.pdf_folder, pdf))
                pages = loader.load()

                # Extract metadata from filename
                base = pdf.replace(".pdf", "")
                parts = re.split(r"[_\-]", base)
                case_type = "_".join(parts[:-1]) if len(parts) >= 2 else base
                case_year = parts[-1] if parts[-1].isdigit() else "unknown"

                for page in pages:
                    page.metadata["source_file"] = pdf
                    page.metadata["case_number"] = base
                    page.metadata["case_year"] = case_year
                    page.metadata["case_type"] = case_type

                    # Do NOT prepend headers inside text (ruins embeddings)
                    # Just keep metadata clean.

                all_docs.extend(pages)

            except Exception as e:
                print(f"❌ Error loading {pdf}: {e}")

        print(f"Loaded {len(all_docs)} pages. Splitting...")

        # ---------------------------------------------------
        # SPLIT TEXT
        # ---------------------------------------------------
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200,
            separators=["\n\n", "\n", ".", " ", ""]
        )
        chunks = splitter.split_documents(all_docs)

        print(f"✂️ Split into {len(chunks)} chunks.")
        print("🔮 Generating embeddings...")

        # ---------------------------------------------------
        # BUILD CHROMA DB
        # ---------------------------------------------------
        self.vectorstore = Chroma.from_documents(
            documents=chunks,
            embedding=self.embeddings,
            persist_directory=self.db_path
        )

        print(f"✅ Saved Chroma DB at {self.db_path}")
        print("Building BM25 index...")

        # ---------------------------------------------------
        # BUILD BM25 CORPUS
        # ---------------------------------------------------
        for ch in chunks:
            self.bm25_docs.append(ch)
            self.bm25_corpus.append(ch.page_content.split())

        self.bm25 = BM25Okapi(self.bm25_corpus)
        print("📌 BM25 index ready.\n")


    # =======================================================================
    #                         CASE NUMBER DETECTION
    # =======================================================================

    def _detect_case_number(self, query: str) -> Optional[str]:
        """
        Detect: CPLA 210 of 2024, C.A.981_2018, Cr.A 52/2021, etc.
        """
        pattern = r"(CPLA|C\.A\.|Cr\.A|C\.P\.|HCA|RFA)[\s\-]*\d+[\s/]*(?:of\s*)?\d{4}"
        match = re.search(pattern, query, re.IGNORECASE)
        if match:
            case = match.group(0)
            case = case.replace(" ", "_").replace("of_", "_").replace("/", "_")
            case = re.sub(r"__+", "_", case)
            return case
        return None


    # =======================================================================
    #                         HYBRID RETRIEVER
    # =======================================================================

    def _retrieve_documents(self, query: str, k: int = 20) -> List[Dict]:
        hits = self.vectorstore.similarity_search(query, k=k)
    
        results = []
        for doc in hits:
            results.append({
                "content": doc.page_content,
                "source": doc.metadata.get("source_file", "unknown"),
                "case_number": doc.metadata.get("case_number", "unknown"),
                "case_year": doc.metadata.get("case_year", "unknown"),
                "case_type": doc.metadata.get("case_type", "unknown"),
                "score": 1.0
            })
        return results
    


    # =======================================================================
    #                         ANSWER GENERATOR
    # =======================================================================

    def _generate_answer(self, query: str, docs: List[Dict]) -> str:
        """
        If LLM available → summarize.
        Otherwise → extractive answer.
        """

        if not self.llm or not docs:
            return self._simple_answer(docs)

        context = "\n\n".join(
            f"[{d['source']}] {d['content'][:500]}" for d in docs[:5]
        )

        prompt = f"""
You are a legal AI assistant summarizing Pakistan Supreme Court judgments.
Use ONLY the excerpts below. NEVER guess or invent.

QUERY: {query}

EXCERPTS:
{context}

Give a concise answer based strictly on the excerpts.
"""

        try:
            out = self.llm.invoke(prompt)
            if isinstance(out, list) and len(out) > 0 and isinstance(out[0], dict):
                return out[0].get("generated_text", str(out))
            return out if isinstance(out, str) else str(out)
        except:
            return self._simple_answer(docs)

    def _simple_answer(self, docs: List[Dict]):
        ans = "Based on retrieved excerpts:\n\n"
        for d in docs[:5]:
            ans += f"{d['source']} → {d['content'][:300]}\n\n"
        return ans


    # =======================================================================
    #                         MAIN SEARCH
    # =======================================================================

    def search(self, query: str, k: int = 20) -> Dict:
        print(f"\n🔍 QUERY: {query}")

        case_num = self._detect_case_number(query)
        if case_num:
            print(f"📋 Detected case number: {case_num}")
        
        docs = self._retrieve_documents(query, k=k)

        print("\n📄 Retrieved:")
        for d in docs:
            print(" -", d["source"])

        answer = self._generate_answer(query, docs)

        return {
            "answer": answer,
            "documents": docs,
            "sources": list({d["source"] for d in docs}),
            "relevant_pdfs": list({d["source"] for d in docs})
        }

In [7]:
PDF_FOLDER = "/kaggle/input/fyp-data/supreme_court_judgments"

print("Starting LOCAL Legal Search Agent\n")
print("No API keys, No Ollama - runs 100% locally!\n")

agent = LegalSearchAgent(
    pdf_folder=PDF_FOLDER,
    test_mode=False
)

agent.build_vectordb()

print("\n" + "="*80)
print("LEGAL SEARCH AGENT READY (LOCAL)")
print("="*80)


Starting LOCAL Legal Search Agent

No API keys, No Ollama - runs 100% locally!

🤖 Loading local LLM (Phi-2)...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

📄 Loading [750/974]: Crl.P.L.A.887-L_2013.pdf
📄 Loading [751/974]: C.A.377_2014.pdf
📄 Loading [752/974]: C.P.L.A.3062_2022.pdf
📄 Loading [753/974]: J.P.195_2017.pdf
📄 Loading [754/974]: C.M.A.12587_2021.pdf
📄 Loading [755/974]: C.P.L.A.546_2021.pdf
📄 Loading [756/974]: C.M.Appeal.39_2021.pdf
📄 Loading [757/974]: C.P.L.A.3300_2024.pdf
📄 Loading [758/974]: Crl.P.L.A.952_2021.pdf
📄 Loading [759/974]: Crl.P.L.A.1602_2023.pdf
📄 Loading [760/974]: Crl.P.L.A.668_2019.pdf
📄 Loading [761/974]: J.P.50_2023.pdf
📄 Loading [762/974]: J.P.252_2020.pdf
📄 Loading [763/974]: C.A.647_2018.pdf
📄 Loading [764/974]: Crl.A.379_2021.pdf
📄 Loading [765/974]: C.P.L.A.159_2021.pdf
📄 Loading [766/974]: C.P.L.A.394-P_2010.pdf
📄 Loading [767/974]: C.P.L.A.559-P_2024.pdf
📄 Loading [768/974]: C.P.L.A.1692-L_2020.pdf
📄 Loading [769/974]: Crl.P.L.A.1075-L_2020.pdf
📄 Loading [770/974]: Crl.P.L.A.69-Q_2022.pdf
📄 Loading [771/974]: C.P.L.A.3179-L_2023.pdf
📄 Loading [772/974]: C.A.17-Q_2023.pdf
📄 Loading [773/974]: S.M.C.

📄 Loading [775/974]: C.P.L.A.3644_2020.pdf
📄 Loading [776/974]: C.P.L.A.1290-L_2019.pdf
📄 Loading [777/974]: C.A.1414_2013.pdf
📄 Loading [778/974]: C.P.L.A.3984_2024.pdf
📄 Loading [779/974]: C.P.L.A.109-L_2024.pdf
📄 Loading [780/974]: Crl.P.L.A.231_2021.pdf
📄 Loading [781/974]: Crl.P.L.A.1690-L_2016.pdf
📄 Loading [782/974]: C.P.L.A.2987-L_2019.pdf
📄 Loading [783/974]: J.P.516_2018.pdf
📄 Loading [784/974]: Crl.A.91_2024.pdf
📄 Loading [785/974]: C.P.L.A.2537_2020.pdf
📄 Loading [786/974]: Crl.P.L.A.146_2025.pdf
📄 Loading [787/974]: Crl.A.238_2021.pdf
📄 Loading [788/974]: C.A.1444_2013.pdf
📄 Loading [789/974]: C.A.700_2014.pdf
📄 Loading [790/974]: C.A.1692_2021.pdf
📄 Loading [791/974]: C.P.L.A.2790_2018.pdf
📄 Loading [792/974]: Crl.P.L.A.1117_2024.pdf
📄 Loading [793/974]: C.P.L.A.4389_2023.pdf
📄 Loading [794/974]: C.P.L.A.414_2021.pdf
📄 Loading [795/974]: C.P.L.A.5666_2024.pdf
📄 Loading [796/974]: C.A.725_2008.pdf
📄 Loading [797/974]: C.A.875_2017.pdf
📄 Loading [798/974]: C.P.21_2022.pdf
📄

📄 Loading [829/974]: C.P.L.A.2743_2017.pdf
❌ Error loading C.P.L.A.2743_2017.pdf: Invalid Elementary Object starting with b'I' @22327: b'l)3123.3n)19( )98 0.IT\n/F2 12.0 Tf\n 0.0 0.0 rg\n0.9998 137(s)8( )-70(ne2(h)19(e)3'
📄 Loading [830/974]: C.P.L.A.5601_2021.pdf
📄 Loading [831/974]: C.A.364_2023.pdf
📄 Loading [832/974]: J.P.14_2020.pdf
📄 Loading [833/974]: C.P.L.A.1809_2020.pdf
📄 Loading [834/974]: Crl.A.525_2022.pdf
📄 Loading [835/974]: Crl.P.L.A.1016-L_2021.pdf
📄 Loading [836/974]: Crl.A.425_2019.pdf
📄 Loading [837/974]: Crl.A.36_2023.pdf
📄 Loading [838/974]: C.P.L.A.1842-L_2022.pdf
📄 Loading [839/974]: C.A.1518_2013.pdf
📄 Loading [840/974]: J.P.42_2017.pdf
📄 Loading [841/974]: Crl.A.199_2023.pdf
📄 Loading [842/974]: Crl.A.322_2018.pdf
📄 Loading [843/974]: C.P.L.A.1618_2024.pdf
📄 Loading [844/974]: C.A.248_2014.pdf
📄 Loading [845/974]: C.P.6_2023.pdf
📄 Loading [846/974]: C.A.138-L_2010.pdf
📄 Loading [847/974]: Crl.P.L.A.54_2023.pdf
📄 Loading [848/974]: Crl.P.L.A.952_2023.pdf
📄 Load

📄 Loading [872/974]: C.P.L.A.1926-L_2015.pdf
📄 Loading [873/974]: C.P.L.A.1010-L_2022.pdf
📄 Loading [874/974]: C.A.550-L_2009.pdf
📄 Loading [875/974]: C.A.151-P_2013.pdf
📄 Loading [876/974]: J.P.23_2023.pdf
📄 Loading [877/974]: Crl.A.314-L_2020.pdf
📄 Loading [878/974]: C.A.1683_2014.pdf
📄 Loading [879/974]: Crl.P.L.A.1124-L_2015.pdf
📄 Loading [880/974]: C.P.L.A.184_2024.pdf
📄 Loading [881/974]: C.A.1227_2016.pdf
📄 Loading [882/974]: C.P.L.A.1737-L_2020.pdf
📄 Loading [883/974]: C.A.350_2020.pdf
📄 Loading [884/974]: C.A.1394_2024.pdf
📄 Loading [885/974]: C.P.L.A.1422-L_2021.pdf
📄 Loading [886/974]: C.P.L.A.2478_2024.pdf
📄 Loading [887/974]: Crl.P.L.A.150-K_2024.pdf
📄 Loading [888/974]: Crl.P.L.A.435_2021.pdf
📄 Loading [889/974]: C.P.L.A.4177_2024.pdf
📄 Loading [890/974]: C.P.L.A.2475-L_2024.pdf
📄 Loading [891/974]: C.A.2434_2016.pdf
📄 Loading [892/974]: Crl.P.L.A.660_2024.pdf
📄 Loading [893/974]: C.A.700_2016.pdf
📄 Loading [894/974]: C.P.L.A.473-K_2023.pdf
📄 Loading [895/974]: C.P.L.A.11

📄 Loading [910/974]: Crl.A.81-L_2017.pdf
📄 Loading [911/974]: C.P.L.A.4618_2019.pdf
📄 Loading [912/974]: C.P.L.A.2414-L_2015.pdf
📄 Loading [913/974]: Crl.P.L.A.1288-L_2017.pdf
📄 Loading [914/974]: Crl.P.L.A.230_2019.pdf
📄 Loading [915/974]: J.P.611_2022.pdf
📄 Loading [916/974]: C.P.L.A.183_2024.pdf
📄 Loading [917/974]: C.M.A.3610_2022.pdf
📄 Loading [918/974]: C.R.P.255_2021.pdf
📄 Loading [919/974]: C.A.8-Q_2017.pdf
📄 Loading [920/974]: C.R.P.988_2023.pdf
📄 Loading [921/974]: J.P.644_2017.pdf
📄 Loading [922/974]: C.P.L.A.757-L_2021.pdf
📄 Loading [923/974]: C.P.L.A.3116_2022.pdf
📄 Loading [924/974]: C.A.139-P_2013.pdf
📄 Loading [925/974]: Crl.A.304_2020.pdf
📄 Loading [926/974]: Crl.P.L.A.806_2022.pdf
📄 Loading [927/974]: C.P.L.A.3127_2020.pdf
📄 Loading [928/974]: C.A.24-Q_2014.pdf
📄 Loading [929/974]: Crl.A.144-L_2020.pdf
📄 Loading [930/974]: Crl.A.92-L_2017.pdf
📄 Loading [931/974]: C.P.L.A.2400-L_2022.pdf
📄 Loading [932/974]: Crl.P.L.A.344_2018.pdf
📄 Loading [933/974]: C.P.L.A.1189_2025

In [ ]:
print("\nType your query (or 'quit' to exit)")
print("Example: 'Why did the court reject SIC's appeal?'\n")

while True:
    query = input("\n💬 Your query: ").strip()
    
    if query.lower() in ['quit', 'exit', 'q']:
        print("\n👋 Goodbye!")
        break
    
    if not query:
        continue
    
    try:
        result = agent.search(query)
        
        print("\n📎 Relevant PDFs:")
        for pdf in result['relevant_pdfs']:
            print(f"   - {pdf}")
        print()
    except Exception as e:
        print(f"❌ Error: {e}")
        print("Please try again or check your setup.\n")


Type your query (or 'quit' to exit)
Example: 'Why did the court reject SIC's appeal?'




💬 Your query:  What was CPLA 210 of 2024 about?


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



🔍 QUERY: What was CPLA 210 of 2024 about?
📋 Detected case number: CPLA_210_2024

📄 Retrieved:
 - C.P.L.A.1573_2024.pdf
 - C.A.1509_2021.pdf
 - C.P.L.A.3297_2024.pdf
 - C.A.2026_2022.pdf
 - C.P.L.A.6-L_2023.pdf
 - C.P.L.A.1618_2024.pdf
 - C.P.L.A.3578_2024.pdf
 - C.P.L.A.5516_2024.pdf
 - C.P.L.A.47_2024.pdf
 - C.P.L.A.694-P_2024.pdf
 - C.P.L.A.767_2022.pdf
 - C.P.L.A.2477_2024.pdf

📎 Relevant PDFs:
   - C.P.L.A.3297_2024.pdf
   - C.A.1509_2021.pdf
   - C.A.2026_2022.pdf
   - C.P.L.A.1573_2024.pdf
   - C.P.L.A.47_2024.pdf
   - C.P.L.A.767_2022.pdf
   - C.P.L.A.694-P_2024.pdf
   - C.P.L.A.1618_2024.pdf
   - C.P.L.A.6-L_2023.pdf
   - C.P.L.A.3578_2024.pdf
   - C.P.L.A.5516_2024.pdf
   - C.P.L.A.2477_2024.pdf




💬 Your query:  What was CPLA 2477 of the year 2024 about?


...

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



🔍 QUERY: What was Civil appeal 647 of 2018 about?

📄 Retrieved:
 - C.A.647_2018.pdf
 - C.A.1280_2019.pdf
 - C.A.1262_2018.pdf
 - C.P.L.A.1182-L_2018.pdf
 - C.P.L.A.620_2021.pdf
 - C.A.239-L_2018.pdf
 - C.A.634_2018.pdf
 - C.P.L.A.2743_2018.pdf
 - Crl.A.597_2018.pdf
 - C.A.152_2019.pdf
 - C.A.785_2022.pdf
 - C.A.415_2018.pdf

📎 Relevant PDFs:
   - C.P.L.A.1182-L_2018.pdf
   - C.A.152_2019.pdf
   - C.A.1280_2019.pdf
   - C.A.647_2018.pdf
   - C.A.1262_2018.pdf
   - C.A.634_2018.pdf
   - C.P.L.A.620_2021.pdf
   - Crl.A.597_2018.pdf
   - C.A.415_2018.pdf
   - C.A.239-L_2018.pdf
   - C.P.L.A.2743_2018.pdf
   - C.A.785_2022.pdf




💬 Your query:  What was CPLA 210 of 2024 about/


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



🔍 QUERY: What was CPLA 210 of 2024 about/
📋 Detected case number: CPLA_210_2024

📄 Retrieved:
 - C.P.L.A.1573_2024.pdf
 - C.A.1509_2021.pdf
 - C.P.L.A.3297_2024.pdf
 - C.A.2026_2022.pdf
 - C.P.L.A.6-L_2023.pdf
 - C.P.L.A.1618_2024.pdf
 - C.P.L.A.3578_2024.pdf
 - C.P.L.A.5516_2024.pdf
 - C.P.L.A.47_2024.pdf
 - C.P.L.A.694-P_2024.pdf
 - C.P.L.A.767_2022.pdf
 - C.P.L.A.2477_2024.pdf


In [9]:
query = "Civil Petition for Leave to Appeal 2024"
docs = agent.vectorstore.similarity_search_with_score(query, k=10)

# Print results
for doc, score in docs[:5]:
    print(f"Score: {score:.4f}")
    print(f"Source: {doc.metadata['source_file']}")
    print(f"Content: {doc.page_content[:200]}\n")

Score: 27.3124
Source: C.P.L.A.2975_2024.pdf
Content: Court. Accordingly, the instant Petition is dismissed and Leave to 
Appeal is refused.  
 
  
 
 
Judge 
 
 
 
Judge 
Islamabad  
04.09.2025 
APPROVED FOR REPORTING 
Younus Shaikh, LC 
 
 
 
1 2002 SC

Score: 27.9933
Source: C.P.L.A.1540_2018.pdf
Content: petitioners has not been able to make out a case for the grant of
leave.
Thus, for the foregoing reasons,
dismissed and leave to appeal is refused.
rm-
El
this petition is
Islamabad,
11.02.2022
APPROV

Score: 28.9388
Source: C.P.L.A.34-Q_2019.pdf
Content: Civil Petition No. 34-Q of 2019 
- 6 -
present petition being bereft of merit is dismissed, and leave to 
appeal is declined. 
 
Judge 
 
 
Judge 
Quetta 
20.05.2022 
Approved For Reporting 
Arif

Score: 29.8974
Source: C.P.L.A.2477_2024.pdf
Content: misinterpreted or not considered.” 
 For the above reasons, while refusing leave to appeal , the petition is 
dismissed.  
Judge 
 
 
Judge 
Islamabad 
21.08.2024 
Atif * 
APPROVED FO

In [11]:
# Try more specific queries
specific_queries = [
    "income tax ordinance section 151",
    "writ petition not maintainable",
    "regularization notification 2011",
    "respondents residence High Court"
]

for q in specific_queries:
    docs = agent.vectorstore.similarity_search_with_score(q, k=5)
    print(f"\n{'='*70}")
    print(f"Query: {q}")
    print('='*70)
    for doc, score in docs:
        print(f"Score: {score:.4f} | {doc.metadata['source_file']}")
        print(f"  → {doc.page_content[:150]}...\n")


Query: income tax ordinance section 151
Score: 39.9712 | Crl.P.L.A.134_2024.pdf
  → Criminal Petition No.134 of 2024 9...

Score: 39.9857 | Crl.P.L.A.134_2024.pdf
  → Criminal Petition No.134 of 2024 7...

Score: 40.2199 | Crl.P.L.A.134_2024.pdf
  → Criminal Petition No.134 of 2024 6...

Score: 40.3741 | Crl.P.L.A.134_2024.pdf
  → Criminal Petition No.134 of 2024 12...

Score: 41.0174 | Crl.P.L.A.134_2024.pdf
  → Criminal Petition No.134 of 2024 3...


Query: writ petition not maintainable
Score: 40.4742 | C.P.L.A.2975_2024.pdf
  → Court. Accordingly, the instant Petition is dismissed and Leave to 
Appeal is refused.  
 
  
 
 
Judge 
 
 
 
Judge 
Islamabad  
04.09.2025 
APPROVED...

Score: 40.7968 | C.P.L.A.2400-L_2022.pdf
  → 7.   In light of this, the petition is found to be 
without merit and dismissed accordingly. Leave to appeal is 
refused.  
 
   
 
 Judge 
 
 
 
 
Ju...

Score: 41.3981 | C.P.L.A.34-Q_2019.pdf
  → Civil Petition No. 34-Q of 2019 
- 6 -
present petition being b

In [14]:

# Check chunk quality
print("="*70)
print("CHUNK QUALITY DIAGNOSIS")
print("="*70)

# Sample 10 random chunks
import random
random_indices = random.sample(range(len(agent.bm25_docs)), min(10, len(agent.bm25_docs)))

for idx in random_indices:
    doc = agent.bm25_docs[idx]
    content = doc.page_content
    
    print(f"\nChunk from: {doc.metadata.get('source_file', 'unknown')}")
    print(f"Length: {len(content)} chars")
    print(f"Content:\n{content[:300]}\n...")
    print("-" * 70)

# Check if chunks contain actual case details
print("\n" + "="*70)
print("SEARCHING FOR SPECIFIC CONTENT")
print("="*70)

search_terms = [
    "income tax",
    "section 151",
    "writ petition",
    "maintainable",
    "regularization",
    "notification",
    "respondent",
    "residence"
]

for term in search_terms:
    count = sum(1 for doc in agent.bm25_docs if term.lower() in doc.page_content.lower())
    print(f"'{term}': found in {count}/{len(agent.bm25_docs)} chunks ({100*count/len(agent.bm25_docs):.1f}%)")

# Check metadata
print("\n" + "="*70)
print("METADATA CHECK")
print("="*70)

print(f"Total chunks: {len(agent.bm25_docs)}")
print(f"Sample metadata:")
for doc in agent.bm25_docs[:3]:
    print(f"\n  Source: {doc.metadata.get('source_file')}")
    print(f"  Case Number: {doc.metadata.get('case_number')}")
    print(f"  Case Year: {doc.metadata.get('case_year')}")
    print(f"  Case Type: {doc.metadata.get('case_type')}")

CHUNK QUALITY DIAGNOSIS

Chunk from: C.P.L.A.824-K_2023.pdf
Length: 990 chars
Content:
(Against the judgment of High 
Court of Sindh, Karachi dated 
07.2.2023, passed in CP No.D-
8559/2019) 
The Commissioner Inland 
Revenue (Legal) v. Pakistan 
Beverages Limited & others 
and 
(81) 
C.P.L.A.602-K/2023 
(Against the judgment of High 
Court of Sindh, Karachi dated 
07.2.2023, passed in 
...
----------------------------------------------------------------------

Chunk from: Crl.A.297_2023.pdf
Length: 970 chars
Content:
1
)7^r
Muhammad Azam
...Appellant
versus
The Stat etc.
...Respondents
Ms. Shazia Bilal, ASCFor the Appellant:
Mr. Irfan Zia, APG.For the State:
19.02.2025Date of hearing:
JUDGMENT
1
PRESENT:
Mr. Justice Salahuddin Panhwar
Mr. Justice Ishtiaq Ibrahim
Criminal Appeal No. 297 of 2023 
(Against the judg
...
----------------------------------------------------------------------

Chunk from: J.P.431_2016.pdf
Length: 348 chars
Content:
while recording the detailed reason. For inst